In [39]:
!pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable


## 1. Installing Required Libraries

In [51]:
import requests
import pandas as pd
import json
import logging
import time
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## 3. Setting Up Logging`

In [52]:
logging.basicConfig(
    filename="weather_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logger = logging.getLogger(__name__)

logger.info("Weather ETL Pipeline started")

pipeline_start_time = time.time()

## 4. Setting Up API Parameters

In [7]:
params = {
    "latitude": 17.3850,
    "longitude": 78.4867,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation"
    ],
    "forecast_days": 1,
    "timezone": "Asia/Kolkata"
}

print(params)

{'latitude': 17.385, 'longitude': 78.4867, 'hourly': ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation'], 'forecast_days': 1, 'timezone': 'Asia/Kolkata'}


In [9]:
api_url = "https://api.open-meteo.com/v1/forecast"

## 5. Setting Up API Session and Retry Mechanism

In [10]:
session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

logger.info("API retry mechanism configured")

## 6. Extracting Weather Data from API

In [11]:
try:
    response = session.get(
        api_url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    logger.info("API data extracted successfully")

except requests.exceptions.RequestException as e:

    logger.exception(f"API request failed: {e}")
    raise

except ValueError as e:

    logger.exception(f"Invalid JSON response: {e}")
    raise

## 7. Inspecting API Response

In [12]:
data = response.json()
print(data)

{'latitude': 17.398945, 'longitude': 78.457085, 'generationtime_ms': 0.07271766662597656, 'utc_offset_seconds': 19800, 'timezone': 'Asia/Kolkata', 'timezone_abbreviation': 'GMT+5:30', 'elevation': 505.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h', 'precipitation': 'mm'}, 'hourly': {'time': ['2026-11-07T00:00', '2026-11-07T01:00', '2026-11-07T02:00', '2026-11-07T03:00', '2026-11-07T04:00', '2026-11-07T05:00', '2026-11-07T06:00', '2026-11-07T07:00', '2026-11-07T08:00', '2026-11-07T09:00', '2026-11-07T10:00', '2026-11-07T11:00', '2026-11-07T12:00', '2026-11-07T13:00', '2026-11-07T14:00', '2026-11-07T15:00', '2026-11-07T16:00', '2026-11-07T17:00', '2026-11-07T18:00', '2026-11-07T19:00', '2026-11-07T20:00', '2026-11-07T21:00', '2026-11-07T22:00', '2026-11-07T23:00'], 'temperature_2m': [23.8, 23.4, 23.0, 22.7, 22.3, 22.7, 22.3, 23.4, 25.2, 27.0, 28.4, 29.4, 30.3, 30.7, 31.0, 30.5, 29.8, 28.8, 27.8, 27.0, 26.3, 25.5, 24.7,

In [13]:
print(data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [14]:
print(data["hourly"])

{'time': ['2026-11-07T00:00', '2026-11-07T01:00', '2026-11-07T02:00', '2026-11-07T03:00', '2026-11-07T04:00', '2026-11-07T05:00', '2026-11-07T06:00', '2026-11-07T07:00', '2026-11-07T08:00', '2026-11-07T09:00', '2026-11-07T10:00', '2026-11-07T11:00', '2026-11-07T12:00', '2026-11-07T13:00', '2026-11-07T14:00', '2026-11-07T15:00', '2026-11-07T16:00', '2026-11-07T17:00', '2026-11-07T18:00', '2026-11-07T19:00', '2026-11-07T20:00', '2026-11-07T21:00', '2026-11-07T22:00', '2026-11-07T23:00'], 'temperature_2m': [23.8, 23.4, 23.0, 22.7, 22.3, 22.7, 22.3, 23.4, 25.2, 27.0, 28.4, 29.4, 30.3, 30.7, 31.0, 30.5, 29.8, 28.8, 27.8, 27.0, 26.3, 25.5, 24.7, 24.2], 'relative_humidity_2m': [77, 78, 78, 77, 76, 83, 84, 78, 64, 52, 47, 44, 42, 40, 39, 40, 41, 45, 50, 53, 53, 57, 60, 59], 'wind_speed_10m': [5.6, 5.0, 6.5, 7.4, 6.8, 5.0, 5.2, 4.0, 5.1, 6.7, 7.3, 7.6, 7.6, 8.1, 8.9, 10.2, 9.8, 11.9, 11.6, 8.6, 8.0, 7.2, 6.3, 6.1], 'precipitation': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

## 8. Converting JSON Data into Pandas DataFrame

In [15]:
try:
    df = pd.DataFrame(data["hourly"])

    logger.info(
        f"DataFrame created successfully. Records: {len(df)}"
    )

except (KeyError, TypeError, ValueError) as e:

    logger.exception(f"DataFrame creation failed: {e}")
    raise

## 9. Exploring the Weather Dataset

In [16]:
print("Dataset Shape:", df.shape)

Dataset Shape: (24, 5)


In [17]:
print(df.columns)

Index(['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
       'precipitation'],
      dtype='object')


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  24 non-null     object 
 1   temperature_2m        24 non-null     float64
 2   relative_humidity_2m  24 non-null     int64  
 3   wind_speed_10m        24 non-null     float64
 4   precipitation         24 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 1.1+ KB


In [19]:
print(df.isnull().sum())

time                    0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
precipitation           0
dtype: int64


In [20]:
print(df.duplicated().sum())

0


In [21]:
print(df.describe())

       temperature_2m  relative_humidity_2m  wind_speed_10m  precipitation
count       24.000000             24.000000       24.000000           24.0
mean        26.258333             59.041667        7.354167            0.0
std          3.032421             15.815455        2.054894            0.0
min         22.300000             39.000000        4.000000            0.0
25%         23.400000             44.750000        5.975000            0.0
50%         25.900000             55.000000        7.250000            0.0
75%         28.950000             77.000000        8.225000            0.0
max         31.000000             84.000000       11.900000            0.0


## 10. Validating Weather Data

In [22]:
try:
    logger.info("Data validation started")

    missing_values = df.isnull().sum().sum()
    duplicate_records = df.duplicated().sum()

    logger.info(f"Missing values: {missing_values}")
    logger.info(f"Duplicate records: {duplicate_records}")
    logger.info(f"Total records validated: {len(df)}")

    logger.info("Data validation completed successfully")

except Exception as e:

    logger.exception(f"Data validation failed: {e}")
    raise

## 11. Transforming Weather Data

In [23]:
try:
    df["time"] = pd.to_datetime(df["time"])

    logger.info("Datetime transformation completed")

except (KeyError, ValueError, TypeError) as e:

    logger.exception(f"Data transformation failed: {e}")
    raise

## 12. Exporting Processed Weather Data

In [24]:
try:
    with open("api_response.json", "w") as file:
        json.dump(data, file, indent=4)

    logger.info("JSON export completed successfully")

    df.to_csv("hyderabad_weather_data.csv", index=False)

    logger.info("CSV export completed successfully")

except (OSError, TypeError, ValueError) as e:

    logger.exception(f"Data export failed: {e}")
    raise

In [25]:
df.to_csv("hyderabad_weather_data.csv",index=False)

## 13. Monitoring Pipeline Execution Time

In [26]:
pipeline_end_time = time.time()

execution_time = pipeline_end_time - pipeline_start_time

logger.info("Weather ETL Pipeline completed successfully")

logger.info(f"Total records processed: {len(df)}")

logger.info(f"Total execution time: {execution_time:.2f} seconds")

print("Pipeline Status: SUCCESS")
print(f"Total Records Processed: {len(df)}")
print(f"Execution Time: {execution_time:.2f} seconds")

Pipeline Status: SUCCESS
Total Records Processed: 24
Execution Time: 761.70 seconds


## 14. Verifying Processed Weather Dataset

In [27]:
df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation
0,2026-11-07 00:00:00,23.8,77,5.6,0.0
1,2026-11-07 01:00:00,23.4,78,5.0,0.0
2,2026-11-07 02:00:00,23.0,78,6.5,0.0
3,2026-11-07 03:00:00,22.7,77,7.4,0.0
4,2026-11-07 04:00:00,22.3,76,6.8,0.0


In [28]:
df.dtypes

time                    datetime64[ns]
temperature_2m                 float64
relative_humidity_2m             int64
wind_speed_10m                 float64
precipitation                  float64
dtype: object

## 15. Installing Database Connectivity Library

In [29]:
!pip install pyodbc

Defaulting to user installation because normal site-packages is not writeable


## 16. Checking Available ODBC Drivers

In [30]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [31]:
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [32]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-JA5ED2F\\SQLEXPRESS;"
    "DATABASE=WeatherData;"
    "Trusted_Connection=yes;"
)

In [33]:
conn = pyodbc.connect(connection_string)

print("Successfully connected to SQL Server!")

Successfully connected to SQL Server!


## 17. Verifying SQL Server Database Connection

In [34]:
cursor = conn.cursor()

cursor.execute("SELECT DB_NAME()")

print("Connected Database:", cursor.fetchone()[0])

Connected Database: WeatherData


## 18. Setting Up SQLAlchemy Database Engine

In [35]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [36]:
connection_url = URL.create(
    "mssql+pyodbc",
    query={
        "odbc_connect": connection_string
    }
)

engine = create_engine(connection_url)

## 19. Loading Weather Data into SQL Server

In [37]:
df.to_sql(
    name="WeatherHourlyData",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print("Weather data successfully loaded into SQL Server!")

Weather data successfully loaded into SQL Server!


## 20. Verifying Data Loading into SQL Server

In [54]:
cursor.execute("SELECT COUNT(*) FROM dbo.WeatherHourlyData")

print("Total Records in SQL Server:", cursor.fetchone()[0])

Total Records in SQL Server: 24


## 21. Final Pipeline Summary